In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from timm.data import resolve_model_data_config, create_transform
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("Using device:", device)

LOCAL_RUN_NAME = "model_i13_dinov2_train_aug_frozen"

BASE_DIR = Path.cwd().parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"

SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
OUTPUT_DIR_BASE = BASE_DIR / "outputs" / "image_modeling" / LOCAL_RUN_NAME
MODEL_DIR_BASE = BASE_DIR / "data" / "models" / LOCAL_RUN_NAME

OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)
MODEL_DIR_BASE.mkdir(parents=True, exist_ok=True)

DINO_MODEL_NAME = "vit_small_patch14_dinov2"
BATCH_SIZE = 8
IMAGE_SIZE = 224
INITIAL_LR = 1e-4
MAX_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
WEIGHT_DECAY = 1e-4
ACCUM_STEPS = 1
NUM_WORKERS = 0
MODEL_NAME = "model_i13_dinov2_train_aug_frozen"

TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

BEST_CHECKPOINT = MODEL_DIR_BASE / "best_model_dinov2_small.pt"
LAST_CHECKPOINT = MODEL_DIR_BASE / "last_checkpoint_dinov2_small.pt"

HISTORY_CSV = OUTPUT_DIR_BASE / "history.csv"
VAL_PRED_CSV = OUTPUT_DIR_BASE / "val_predictions.csv"
FUSION_TABLE_CSV = OUTPUT_DIR_BASE / "val_fusion_table.csv"
CLASSIFICATION_REPORT_CSV = OUTPUT_DIR_BASE / "classification_report.csv"
RUN_METADATA_JSON = OUTPUT_DIR_BASE / "run_metadata.json"

LOGITS_NPY = OUTPUT_DIR_BASE / "y_logits.npy"
PROBA_NPY = OUTPUT_DIR_BASE / "y_proba.npy"

CONFUSION_MATRIX_PNG = OUTPUT_DIR_BASE / "confusion_matrix.png"
TRAIN_LOSS_PNG = OUTPUT_DIR_BASE / "train_loss.png"
TRAIN_ACC_PNG = OUTPUT_DIR_BASE / "training_accuracy.png"
TRAIN_F1_PNG = OUTPUT_DIR_BASE / "training_macro_f1.png"

print("BASE_DIR:", BASE_DIR)
print("TRAIN_SPLIT_PATH exists:", TRAIN_SPLIT_PATH.exists())
print("VAL_SPLIT_PATH exists:", VAL_SPLIT_PATH.exists())
print("LABEL2ID_PATH exists:", LABEL2ID_PATH.exists())
print("IMAGE_DIR exists:", IMAGE_DIR.exists())

train_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_df = pd.read_csv(VAL_SPLIT_PATH)

with open(LABEL2ID_PATH, "r") as f:
    raw_label2id = json.load(f)

label2id = {int(k): int(v) for k, v in raw_label2id.items()}
id2label = {v: k for k, v in label2id.items()}

class DinoV2ImageClassifier(nn.Module):
    def __init__(self, model_name, num_classes, freeze_backbone=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        with torch.no_grad():
            dummy_size = self.backbone.default_cfg["input_size"][-1]
            dummy = torch.zeros(1, 3, dummy_size, dummy_size)
            feats = self.backbone.forward_features(dummy)
            feats = self.backbone.forward_head(feats, pre_logits=True)
            in_features = feats.shape[-1]

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, image):
        feats = self.backbone.forward_features(image)
        feats = self.backbone.forward_head(feats, pre_logits=True)
        logits = self.classifier(feats)
        return logits

model = DinoV2ImageClassifier(
    model_name=DINO_MODEL_NAME,
    num_classes=len(label2id),
    freeze_backbone=True
).to(device)

data_cfg = resolve_model_data_config(model.backbone)
print("Resolved model data config:", data_cfg)

train_transform = create_transform(
    input_size=data_cfg["input_size"],
    mean=data_cfg["mean"],
    std=data_cfg["std"],
    interpolation=data_cfg["interpolation"],
    crop_pct=data_cfg.get("crop_pct", 1.0),
    is_training=True
)

val_transform = create_transform(
    input_size=data_cfg["input_size"],
    mean=data_cfg["mean"],
    std=data_cfg["std"],
    interpolation=data_cfg["interpolation"],
    crop_pct=data_cfg.get("crop_pct", 1.0),
    is_training=False
)

class RakutenDinoDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_dir, label_mapping, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.label_mapping = label_mapping
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_dir / f"image_{row['imageid']}_product_{row['productid']}.jpg"
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        label_id = self.label_mapping[int(row["prdtypecode"])]

        return {
            "image": image,
            "labels": torch.tensor(label_id, dtype=torch.long),
            "imageid": row["imageid"],
            "productid": row["productid"],
            "prdtypecode_raw": int(row["prdtypecode"])
        }

def collate_fn(batch):
    return {
        "image": torch.stack([x["image"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "imageid": [x["imageid"] for x in batch],
        "productid": [x["productid"] for x in batch],
        "prdtypecode_raw": [x["prdtypecode_raw"] for x in batch]
    }

train_dataset = RakutenDinoDataset(train_df, IMAGE_DIR, label2id, train_transform)
val_dataset = RakutenDinoDataset(val_df, IMAGE_DIR, label2id, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    collate_fn=collate_fn
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=INITIAL_LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

def plot_history(history_df):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{MODEL_NAME} - Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(TRAIN_LOSS_PNG, dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{MODEL_NAME} - Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(TRAIN_ACC_PNG, dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_f1"], label="Train Macro F1")
    plt.plot(history_df["epoch"], history_df["val_f1"], label="Val Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(f"{MODEL_NAME} - Macro F1")
    plt.legend()
    plt.tight_layout()
    plt.savefig(TRAIN_F1_PNG, dpi=200, bbox_inches="tight")
    plt.show()

def run_one_epoch(model, loader, criterion, optimizer=None, accum_steps=1, phase="train"):
    train_mode = optimizer is not None

    if train_mode:
        model.train()
        optimizer.zero_grad()
    else:
        model.eval()

    running_loss = 0.0
    all_preds = []
    all_true = []
    val_rows = []
    all_logits = []
    all_proba = []

    for step, batch in enumerate(tqdm(loader, desc=phase, leave=False), start=1):
        images = batch["image"].to(device)
        labels = batch["labels"].to(device)

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, labels)

            if train_mode:
                (loss / accum_steps).backward()
                if step % accum_steps == 0 or step == len(loader):
                    optimizer.step()
                    optimizer.zero_grad()

        running_loss += loss.item() * labels.size(0)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

        if not train_mode:
            batch_logits = logits.detach().cpu().numpy()
            batch_proba = probs.detach().cpu().numpy()

            all_logits.append(batch_logits)
            all_proba.append(batch_proba)

            for j in range(labels.size(0)):
                pred_label_id = int(preds[j].detach().cpu().item())
                true_label_id = int(labels[j].detach().cpu().item())

                row_dict = {
                    "imageid": batch["imageid"][j],
                    "productid": batch["productid"][j],
                    "true_label_id": true_label_id,
                    "pred_label_id": pred_label_id,
                    "true_prdtypecode": int(id2label[true_label_id]),
                    "pred_prdtypecode": int(id2label[pred_label_id]),
                    "confidence": float(batch_proba[j].max())
                }

                for c in range(batch_logits.shape[1]):
                    row_dict[f"logit_{c}"] = float(batch_logits[j, c])
                    row_dict[f"proba_{c}"] = float(batch_proba[j, c])

                val_rows.append(row_dict)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_f1 = f1_score(all_true, all_preds, average="macro")

    if not train_mode:
        all_logits = np.concatenate(all_logits, axis=0)
        all_proba = np.concatenate(all_proba, axis=0)
    else:
        all_logits = None
        all_proba = None

    return epoch_loss, epoch_acc, epoch_f1, all_true, all_preds, val_rows, all_logits, all_proba

start_epoch = 1
history = []
best_f1 = -1.0
epochs_without_improvement = 0
resume_path = None

if BEST_CHECKPOINT.exists():
    resume_path = BEST_CHECKPOINT
elif LAST_CHECKPOINT.exists():
    resume_path = LAST_CHECKPOINT

if resume_path is not None:
    print(f"Resuming from checkpoint: {resume_path}")

    ckpt = torch.load(resume_path, map_location=device)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    history = ckpt.get("history", [])
    best_f1 = float(ckpt.get("best_f1", -1.0))
    epochs_without_improvement = int(ckpt.get("epochs_without_improvement", 0))
    start_epoch = int(ckpt.get("epoch", 0)) + 1

    print(f"Loaded epoch: {ckpt.get('epoch', 0)}")
    print(f"Next epoch: {start_epoch}")
    print(f"Best val F1 so far: {best_f1:.4f}")
else:
    print("No checkpoint found. Starting from scratch.")

test_batch = next(iter(train_loader))
with torch.no_grad():
    test_logits = model(test_batch["image"].to(device))
print("Sanity check logits:", test_logits.shape)

start_time = time.time()

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    epoch_t0 = time.time()

    train_loss, train_acc, train_f1, _, _, _, _, _ = run_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        accum_steps=ACCUM_STEPS,
        phase="train"
    )

    val_loss, val_acc, val_f1, y_true, y_pred, val_rows, y_logits, y_proba = run_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        optimizer=None,
        accum_steps=1,
        phase="val"
    )

    scheduler.step(val_f1)

    epoch_minutes = (time.time() - epoch_t0) / 60.0
    current_lr = optimizer.param_groups[0]["lr"]

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "lr": current_lr,
        "epoch_minutes": epoch_minutes
    }
    history.append(epoch_record)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | train_f1={train_f1:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | "
        f"lr={current_lr:.2e} | time={epoch_minutes:.2f} min",
        flush=True
    )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_f1": best_f1,
        "epochs_without_improvement": epochs_without_improvement
    }

    torch.save(checkpoint, LAST_CHECKPOINT)

    if val_f1 > best_f1:
        best_f1 = val_f1
        epochs_without_improvement = 0
        checkpoint["best_f1"] = best_f1
        checkpoint["epochs_without_improvement"] = epochs_without_improvement
        torch.save(checkpoint, BEST_CHECKPOINT)

        pd.DataFrame(val_rows).to_csv(VAL_PRED_CSV, index=False)
        pd.DataFrame(val_rows).to_csv(FUSION_TABLE_CSV, index=False)
        np.save(LOGITS_NPY, y_logits)
        np.save(PROBA_NPY, y_proba)

        print(f"Best model saved at epoch {epoch} with val_f1={val_f1:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

total_minutes = (time.time() - start_time) / 60.0

best_source = BEST_CHECKPOINT if BEST_CHECKPOINT.exists() else LAST_CHECKPOINT
best_ckpt = torch.load(best_source, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

best_val_loss, best_val_acc, best_val_f1, y_true, y_pred, val_rows, y_logits, y_proba = run_one_epoch(
    model=model,
    loader=val_loader,
    criterion=criterion,
    optimizer=None,
    accum_steps=1,
    phase="val_final"
)

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_CSV, index=False)

val_pred_df = pd.DataFrame(val_rows)
val_pred_df.to_csv(VAL_PRED_CSV, index=False)
val_pred_df.to_csv(FUSION_TABLE_CSV, index=False)

np.save(LOGITS_NPY, y_logits)
np.save(PROBA_NPY, y_proba)

report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(CLASSIFICATION_REPORT_CSV)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title(f"{MODEL_NAME} - Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PNG, dpi=200, bbox_inches="tight")
plt.show()

plot_history(history_df)

best_epoch_from_history = int(history_df.sort_values("val_f1", ascending=False).iloc[0]["epoch"])

run_metadata = {
    "run_name": LOCAL_RUN_NAME,
    "device": str(device),
    "dino_model_name": DINO_MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "initial_lr": INITIAL_LR,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "weight_decay": WEIGHT_DECAY,
    "accum_steps": ACCUM_STEPS,
    "num_workers": NUM_WORKERS,
    "num_classes": len(label2id),
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "best_val_accuracy": float(best_val_acc),
    "best_val_macro_f1": float(best_val_f1),
    "best_epoch": best_epoch_from_history,
    "total_training_minutes_this_session": float(total_minutes),
    "fusion_table_csv": str(FUSION_TABLE_CSV),
    "logits_npy": str(LOGITS_NPY),
    "proba_npy": str(PROBA_NPY)
}

with open(RUN_METADATA_JSON, "w") as f:
    json.dump(run_metadata, f, indent=2)

print("\nFinal validation results")
print(f"Best Val Accuracy: {best_val_acc:.4f}")
print(f"Best Val Macro F1: {best_val_f1:.4f}")
print(f"Best Epoch From History: {best_epoch_from_history}")
print(f"Training time this session: {total_minutes:.2f} min")
print(f"History: {HISTORY_CSV}")
print(f"Predictions: {VAL_PRED_CSV}")
print(f"Fusion table: {FUSION_TABLE_CSV}")
print(f"Logits: {LOGITS_NPY}")
print(f"Probabilities: {PROBA_NPY}")
print(f"Classification report: {CLASSIFICATION_REPORT_CSV}")
print(f"Metadata: {RUN_METADATA_JSON}")
print(f"Best checkpoint: {BEST_CHECKPOINT}")
print(f"Last checkpoint: {LAST_CHECKPOINT}")
print(f"Figures: {OUTPUT_DIR_BASE}")